This file was used to produce figures for Rae and Ananya's 3MT presentation

In [1]:
import geopandas as gpd
import networkx as nx
import osmnx as ox
import pickle

In [2]:
#loading in the data for the parks intersected with the grid
dallas_county_parks_map = gpd.read_file('Parks/Grid_Intersection_Centroids/grid_intersect.geojson')
#filtering for just the greenbelt
greenbelt = dallas_county_parks_map[dallas_county_parks_map["OBJECTID"] == 1919]
# #checking the crs -- 4326
# greenbelt.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [40]:
#calculating centroids
greenbelt['centroids'] = greenbelt.centroid

#loading the underlying street network so I can find what nodes map to the greenbelt centroids
def load_map(mapp):
    #mapp is the filepath to the multidi graph
    # G = nx.read_gpickle(mapp)
    with open(mapp, 'rb') as f:
        G = pickle.load(f)
    nodes = ox.distance.nearest_nodes(G, greenbelt["centroids"].x, greenbelt["centroids"].y)
    return nodes, G

nodes, G = load_map('Distance Matrix Construction/final_code_files/Dallas_bbox_walk.pkl')

/tmp/ipykernel_2631386/197436177.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  greenbelt['centroids'] = greenbelt.centroid
/users/rtraverfallick/.local/lib/python3.11/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [41]:
nodes = nodes.astype(str)

In [51]:
import numpy as np
park_iso = gpd.read_file('Parks/Grid_Intersection_Centroids/grid_intersect_centroids/isochrones_walk/isochrone_full_walk.geojson')
parks_iso_20 = park_iso[park_iso['eps'] == '20']
# # parks_iso_20[parks_iso_20['node'] == '81730593']
parks_iso_20_all = park_iso[park_iso['eps'] == '20']
parks_iso_20 = parks_iso_20[parks_iso_20['node'].isin(nodes)]

In [52]:
parks_iso_20

,node,eps,geometry
3674,81730593,20,"POLYGON ((-96.79899 32.73672, -96.79921 32.736..."
3680,81905965,20,"POLYGON ((-96.79595 32.7503, -96.79608 32.7503..."
3692,4908159777,20,"POLYGON ((-96.79378 32.73966, -96.79832 32.740..."
3695,4908159777,20,"POLYGON ((-96.79378 32.73966, -96.79832 32.740..."
3851,81906026,20,"POLYGON ((-96.80903 32.74966, -96.81019 32.750..."
...,...,...,...
5600,13404069055,20,"POLYGON ((-96.87362 32.79317, -96.87369 32.793..."
5603,82226913,20,"POLYGON ((-96.84533 32.79863, -96.87908 32.801..."
5606,13076906740,20,"POLYGON ((-96.84055 32.7961, -96.87372 32.8014..."
5609,13404069054,20,"POLYGON ((-96.83684 32.79499, -96.87098 32.802..."


In [50]:
m = parks_iso_20.explore(style_kwds = {'fillOpacity' : 0.05, 'lineWidth': 0.3})
greenbelt['centroids'].explore(m = m, color = 'red')

In [61]:
place = "Dallas County, Texas, United States of America"
api = ox.geocoder.geocode_to_gdf(place)
dallas_geo = api['geometry'].iloc[0]

e = api.explore(
    # color="lightgray",
     # "color": "red",          # Edge color
     #    "weight": 2,             # Edge thickness
     #    "fillColor": "blue",     # Fill color
     #    "fillOpacity": 0.5 
    style_kwds={"color": "black",          # Edge color
        "weight": 2,             # Edge thickness
        "fillColor": "lightgray",     # Fill color
        "fillOpacity": 0.5 } 
# "fillOpacity": 0.5, 'edgeColor': 'black'}
)
parks_iso_20_all.explore(m = e, style_kwds = {'fillOpacity' : 0.9, 'lineWidth': 0.3})